# VoiceStudio on Kaggle â€” Max pack, 2 Ã— T4

Runs the VoiceStudio backend inside a free Kaggle GPU session and exposes it to
the desktop app as a **Remote backend**. Weights come from a private Kaggle
dataset built by `scripts/build_kaggle_dataset.py`; this notebook installs the
code, points the backend at the mounted cache, starts the server and opens a
Cloudflare quick tunnel.

> **Personal, non-commercial use only.** Kaggle's Terms of Use allow no
> commercial use, and the Max pack contains non-commercial weights (NLLB-200 is
> CC-BY-NC, OmniVoice's weights are non-commercial). Keep the dataset **private**.

Before you run:

- the private model dataset is attached (its `hub/` directory is the HF cache root);
- **Settings â†’ Accelerator â†’ GPU T4 Ã—2**;
- **Internet on** (code clone, wheels and the tunnel).

Run the cells top to bottom. The tunnel cell prints the URL and API key to paste
into **Settings â†’ Sharing â†’ Remote backend**.

Guide: `docs/install/kaggle.md` Â· State of the work: `docs/agents/kaggle-refactor-handoff.md`.

In [ ]:
import json
import os
import pathlib
import re
import secrets
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request

REPO_URL = "https://github.com/orivaldogama/SmartStudio.git"
REPO_REF = "main"

WORK = pathlib.Path("/kaggle/working")
SRC = WORK / "voicestudio"
DATA_DIR = WORK / "omnivoice"
PORT = 3900
API_KEY = secrets.token_urlsafe(24)

# The Max model pack, in the order the builder writes it.
MAX_PACK_REPOS = (
    "k2-fsa/OmniVoice",
    "Systran/faster-whisper-large-v3",
    "csukuangfj/sherpa-onnx-whisper-tiny",
    "csukuangfj/sherpa-onnx-nemo-parakeet-tdt-0.6b-v3-int8",
    "facebook/nllb-200-distilled-600M",
)

def find_hub():
    # The dataset's hub/ directory, i.e. the Hugging Face cache root.
    for hub in sorted(pathlib.Path("/kaggle/input").glob("*/hub")):
        if (hub / "models--k2-fsa--OmniVoice").is_dir():
            return hub
    raise SystemExit(
        "No hub/ cache under /kaggle/input. Attach the private model dataset: "
        "right-hand panel â†’ Add Data â†’ Your Datasets."
    )

HUB = find_hub()
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("code:  ", REPO_URL, "@", REPO_REF)
print("cache: ", HUB, "(writable)" if os.access(HUB, os.W_OK) else "(read-only)")
print("data:  ", DATA_DIR, "(ephemeral)")

In [ ]:
gpus = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
rows = [line for line in gpus.stdout.splitlines() if line.strip()]
assert rows, "No GPU visible. Settings â†’ Accelerator â†’ GPU T4 Ã—2, then re-run this cell.\n" + gpus.stderr
print("\n".join(f"  {row}" for row in rows))
print(f"\n{len(rows)} GPU(s). The backend uses device 0 only today; the second card idles")
print("(docs/install/kaggle.md â†’ known gaps).")

## Install the code and the backend

Same install path as the official Docker image and the Colab notebook:
`uv pip install --system` with the repo's own constraints, so torch, torchaudio
and torchvision move together. Expect a few minutes and a few GB of wheels the
first time.

In [ ]:
def run(cmd, *, cwd=None, what=""):
    # Stream a command's output and stop the cell with the cause visible.
    shown = cmd if isinstance(cmd, str) else " ".join(cmd)
    print(f"\n$ {shown}", flush=True)
    proc = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str), text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout, end="", flush=True)
    if proc.returncode != 0:
        raise SystemExit(f"FAILED: {what or shown} (exit {proc.returncode}). Scroll up for the cause.")
    return proc.stdout

# Wheels and model CDNs occasionally stall; uv's default timeout is short.
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

# 1. System packages the audio stack needs (present in most images; kept idempotent).
if not shutil.which("ffmpeg"):
    run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1", what="apt-get install")

# 2. The code, reused when this cell is re-run inside the same session.
if (SRC / ".git").is_dir():
    print("repo already present â€” updating")
    run(["git", "-C", str(SRC), "pull", "--ff-only"], what="git pull")
else:
    run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(SRC)], what="git clone")

# 3. Backend dependencies. uv pip install ignores pyproject's [tool.uv] settings,
#    so the constraints, override and security floors are passed explicitly.
if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], what="pip install uv")
run(["uv", "pip", "install", "--system", "--no-cache",
     "--constraint", "deploy/torch-constraints.txt",
     "--constraint", "deploy/security-constraints.txt",
     "--override", "deploy/uv-overrides.txt", "."],
    cwd=str(SRC), what="backend install (uv pip install --system .)")

floors = run([sys.executable, "deploy/security_floors.py", "requirements"], cwd=str(SRC)).split()
if floors:
    run(["uv", "pip", "install", "--system", "--no-cache",
         "--constraint", "deploy/torch-constraints.txt",
         "--constraint", "deploy/security-constraints.txt",
         "--override", "deploy/uv-overrides.txt", *floors],
        cwd=str(SRC), what="security floor upgrades")
run([sys.executable, "deploy/security_floors.py", "check"], cwd=str(SRC), what="security floor check")

In [ ]:
# cuDNN 8 beside the installed cuDNN 9: CTranslate2 (faster-whisper, WhisperX)
# needs it for GPU transcription. Non-fatal â€” without it ASR falls back to the
# torch-native backend. scripts/setup.py targets <repo>/.venv, so create it.
(SRC / ".venv").mkdir(exist_ok=True)
run([sys.executable, str(SRC / "scripts" / "setup.py")], cwd=str(SRC), what="cuDNN 8 compat")

# Import the backend's own model stack, not just torch: a mismatched trio (the
# "operator torchvision::nms does not exist" failure) only shows up on import.
probe = run([sys.executable, "-c",
             "import torch, torchaudio, transformers;"
             "print('torch', torch.__version__, '| torchaudio', torchaudio.__version__,"
             " '| cuda', torch.cuda.is_available(), '| transformers', transformers.__version__)"],
            cwd=str(SRC), what="import sanity check")
assert "| cuda True" in probe, (
    "PyTorch cannot see the GPU. If it said 'no kernel image is available', the "
    "wheel does not cover this card (sm_75) â€” see the troubleshooting cell."
)

## Start the backend

Headless server mode, loopback bind (the tunnel runs on the same host), an API
key, and `HF_HUB_CACHE` pointing at the read-only dataset. `HF_HUB_OFFLINE` stops
the hub library from trying to repair or re-download anything.

The server actually starts through a small `session_control.py` sidecar this cell
writes: the backend app plus one extra authenticated route,
`POST /api/session-control/shutdown`, so the desktop app can end the session —
and its GPU billing — from **Settings → Sharing → Remote backend**.


In [ ]:
LOG = WORK / "backend.log"
CONTROL = WORK / "session_control.py"
TUNNEL_PID = WORK / "tunnel.pid"

# Session-control sidecar written next to the log: the real backend app plus
# one extra authenticated route, POST /api/session-control/shutdown, behind
# the backend's own Bearer gate. The desktop app calls it from
# Settings -> Sharing -> Remote backend; it then stops the tunnel, ends THIS
# kernel (which ends the Kaggle session and its GPU quota) and exits uvicorn.
SESSION_CONTROL_SOURCE = """# Kaggle session control, written by the bootstrap notebook.
import os
import signal
import sys
import threading
import time

BACKEND_DIR = r"__BACKEND_DIR__"
TUNNEL_PID = r"__TUNNEL_PID__"
if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)

from fastapi.routing import APIRoute  # noqa: E402
from main import app  # noqa: E402
from starlette.responses import JSONResponse  # noqa: E402


def _terminate_from_pidfile(path, expected_cmdline):
    # SIGTERM the pid only while its /proc cmdline still proves its identity.
    try:
        with open(path, encoding="ascii") as handle:
            pid = int(handle.read().strip())
    except (OSError, ValueError):
        return
    try:
        with open("/proc/%d/cmdline" % pid, "rb") as handle:
            cmdline = handle.read().decode("utf-8", "replace")
    except OSError:
        return
    if expected_cmdline not in cmdline:
        return  # stale pid file: never kill an unrelated process
    try:
        os.kill(pid, signal.SIGTERM)
    except OSError:
        pass


def _end_session():
    time.sleep(0.5)  # let the 202 response flush before tearing everything down
    _terminate_from_pidfile(TUNNEL_PID, "cloudflared")
    parent = os.getppid()  # the notebook kernel that spawned this server
    if parent > 1:
        try:
            os.kill(parent, signal.SIGTERM)
            deadline = time.time() + 10
            while time.time() < deadline:
                time.sleep(0.5)
                os.kill(parent, 0)  # raises OSError once the kernel is gone
            os.kill(parent, signal.SIGKILL)  # kernel ignored SIGTERM
        except OSError:
            pass
    os._exit(0)


async def shutdown_session(request):
    authorization = request.headers.get("authorization", "")
    if not authorization.lower().startswith("bearer ") or not authorization[7:].strip():
        return JSONResponse({"detail": "unauthorized"}, status_code=401)
    threading.Thread(target=_end_session, daemon=True).start()
    return JSONResponse({"status": "shutting_down"}, status_code=202)


# Position 0: ahead of the SPA catch-all mount, which would otherwise match
# the path first and answer 404 before the router ever sees the request.
app.router.routes.insert(
    0, APIRoute("/api/session-control/shutdown", shutdown_session, methods=["POST"])
)
"""
CONTROL.write_text(
    SESSION_CONTROL_SOURCE
    .replace("__BACKEND_DIR__", str(SRC / "backend"))
    .replace("__TUNNEL_PID__", str(TUNNEL_PID)),
    encoding="utf-8",
)
env = os.environ.copy()
env.update({
    "OMNIVOICE_SERVER_MODE": "1",
    "OMNIVOICE_PORT": str(PORT),
    "OMNIVOICE_API_KEY": API_KEY,
    "OMNIVOICE_DATA_DIR": str(DATA_DIR),
    "OMNIVOICE_DEVICE": "cuda",
    "OMNIVOICE_ALIGN_DEVICE": "cuda",
    "OMNIVOICE_GPU_WORKERS": "1",
    "HF_HUB_CACHE": str(HUB),
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "PYTHONUNBUFFERED": "1",
})

# Kill a half-dead backend from an earlier run of this cell.
subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
time.sleep(1)

log_handle = LOG.open("w")
backend = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "session_control:app",
     "--app-dir", str(WORK), "--host", "127.0.0.1", "--port", str(PORT)],
    cwd=str(SRC), env=env, stdout=log_handle, stderr=subprocess.STDOUT,
    start_new_session=True,
)
time.sleep(2)
print("backend pid", backend.pid, "â†’", LOG)

In [ ]:
def local(path, timeout=5):
    # GET a path on the local backend; returns (status, body).
    request = urllib.request.Request(f"http://127.0.0.1:{PORT}{path}")
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.status, response.read()


started = time.time()
info = None
while time.time() - started < 600:
    if backend.poll() is not None:
        raise SystemExit(f"backend exited with {backend.returncode}\n\n" + LOG.read_text()[-4000:])
    try:
        status, body = local("/health", timeout=5)
        if status == 200:
            info = json.loads(body)
            break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(3)

print()
if info is None:
    raise SystemExit("backend never answered /health in 10 min; tail:\n\n" + LOG.read_text()[-4000:])
print(f"healthy after {time.time() - started:.0f}s:", info)

## Check the pack is installed

`GET /models` is the same catalogue the desktop app reads, so "installed" here
means the app will not offer a download. Licence acceptance lives in the data
root, which is ephemeral: the app asks once per session.

In [ ]:
status, body = local("/models", timeout=60)
rows = {row.get("hf_repo") or row["repo_id"]: row for row in json.loads(body)["models"]}

missing = []
for repo in MAX_PACK_REPOS:
    row = rows.get(repo)
    if row and row.get("installed"):
        accepted = (row.get("license_acceptance") or {}).get("accepted")
        print(f"  ok    {repo}  {row['size_on_disk_bytes'] / 1024**3:5.2f} GiB  licence accepted: {accepted}")
    else:
        missing.append(repo)
        print(f"  MISS  {repo}  installed={bool(row and row.get('installed'))}")

assert not missing, (
    "the backend does not see these repositories in " + str(HUB) + ": " + ", ".join(missing)
    + "\nAttach the dataset built by scripts/build_kaggle_dataset.py "
      "(pinned revisions, real files, refs/main)."
)

status, body = local("/sysinfo", timeout=30)
info = json.loads(body)
print(f"\ndevice: {info.get('gpu_name')} Â· {info.get('total_vram')} GB Â· "
      f"{info.get('cpu_logical_cores')} vCPU Â· {info.get('total_ram'):.0f} GB RAM")

## Open the tunnel

A Cloudflare quick tunnel dials out, so no inbound port is needed. Its hostname
changes every session â€” paste the new one into the app each time.

In [ ]:
CLOUDFLARED = WORK / "cloudflared"
if not CLOUDFLARED.exists():
    run(["curl", "-fsSL", "-o", str(CLOUDFLARED),
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
        what="cloudflared download")
    CLOUDFLARED.chmod(0o755)

TUNNEL_LOG = WORK / "cloudflared.log"
tunnel_handle = TUNNEL_LOG.open("w")
tunnel = subprocess.Popen(
    [str(CLOUDFLARED), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=tunnel_handle, stderr=subprocess.STDOUT, start_new_session=True,
)

# The session-control sidecar reads this pid file to stop the tunnel.
(WORK / "tunnel.pid").write_text(str(tunnel.pid))

URL_RE = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
PUBLIC_URL = None
deadline = time.time() + 120
while time.time() < deadline and PUBLIC_URL is None:
    time.sleep(2)
    match = URL_RE.search(TUNNEL_LOG.read_text(errors="ignore"))
    PUBLIC_URL = match.group(0) if match else None

if PUBLIC_URL is None:
    raise SystemExit("no tunnel URL after 2 min; see " + str(TUNNEL_LOG))
print("tunnel:", PUBLIC_URL)

In [ ]:
# Any HTTP answer proves the tunnel reaches the backend; 401/403 is fine on /health.
for _attempt in range(10):
    try:
        with urllib.request.urlopen(PUBLIC_URL + "/health", timeout=10) as response:
            print("public /health â†’", response.status)
        break
    except urllib.error.HTTPError as exc:
        print("public /health â†’", exc.code, "(the tunnel is up)")
        break
    except Exception:
        time.sleep(3)
else:
    print("public /health did not answer yet â€” the tunnel may still be warming up")

(WORK / "remote-backend.txt").write_text(f"URL: {PUBLIC_URL}\nAPI key: {API_KEY}\n", encoding="utf-8")

print()
print("=" * 72)
print("Remote backend URL:", PUBLIC_URL)
print("API key:           ", API_KEY)
print("=" * 72)
print("On your computer: Settings â†’ Sharing â†’ Remote backend â†’ paste both")
print("                â†’ Test connection â†’ Save & reload")
print()
print(f"Also saved to {WORK / 'remote-backend.txt'} (gone when the session ends).")

## Keep the session alive

Kaggle ends a session after a stretch without activity. This cell idles and
reprints the connection details. **Interrupt** it when you are done, then run the
cleanup cell â€” the session bills GPU quota while it runs.

The desktop app can also end everything for you: **Settings → Sharing → Remote
backend → Shut down session** stops the tunnel, kills this kernel and releases
the GPU.


In [ ]:
print("Idling keeps the session and the tunnel up. Interrupt this cell to stop it.")
while True:
    print(time.strftime("%H:%M:%S"), "alive â€”", PUBLIC_URL)
    time.sleep(300)

## Stop everything

Nothing in `/kaggle/working` survives unless you *Save Version*, and a saved
version holds only 20 GB. The dataset stays in `/kaggle/input`; voices and
exports made during the session do not.

The app's **Shut down session** button does the same from the other side; this
cell is the manual path inside the Kaggle UI.


In [ ]:
work = globals().get("WORK")
stale = work / "tunnel.pid" if work is not None else None
if stale is not None and stale.exists():
    stale.unlink()

for name, process in (("tunnel", globals().get("tunnel")), ("backend", globals().get("backend"))):
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill()
        print("stopped", name)

log = globals().get("LOG")
if log is not None and log.exists():
    print("\nBackend log tail:\n" + log.read_text()[-1500:])

## Troubleshooting

| Symptom | Cause / fix |
| --- | --- |
| `cuda False`, or `no kernel image is available` | The PyTorch wheel does not cover sm_75. `uv pip install --system torch==<older>+cu128` and re-run the sanity check. |
| `MISS` for every repository | Wrong dataset attached: it must contain `hub/models--k2-fsa--OmniVoice/`. A dataset of loose files (no hub-cache layout) cannot be used. |
| `Permission denied` in `backend.log` | Something tried to write into the read-only cache. `HF_HUB_OFFLINE=1` covers the normal paths; report the endpoint â€” this is the read-only-cache gap in the handoff page. |
| First generate takes minutes or times out | The cold weight load counts against the generate budget. Warm the engine with one short generate before you need it. |
| The app asks to accept a licence every session | Expected: acceptance is stored in the ephemeral data root. Accept once per session. |
| `Test connection` fails from the desktop | The quick-tunnel hostname changed or the tunnel died with the session. Re-run the tunnel cell and paste the new URL. |
| The web UI in a browser gets 403 | A browser carries an `Origin`; add that hostname to `OMNIVOICE_ALLOWED_ORIGINS` and `OMNIVOICE_ALLOWED_HOSTS` and restart the backend. The desktop app needs neither. |
| Dubbing crawls | Dubbing is CPU-bound (demucs, resampling) and only 4 vCPU exist â€” not a GPU problem. |
| Session ended after ~12 h | Kaggle's cap, not an error. Start a new session and re-run. |

Non-commercial only â€” see `docs/licensing/model-inventory.md`.